# Import Libaries

In [47]:
import kagglehub
import pandas as pd

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# 0. Load Data

In [5]:
path = kagglehub.dataset_download(
    "sriharshabsprasad/movielens-dataset-100k-ratings",
    output_dir="data"
)

DATA_PATH = Path(path) / "ml-latest-small"

print("Path to dataset files:", DATA_PATH)

Path to dataset files: data\ml-latest-small


# 1. Data Inspection

## Create paths and dataframes

In [9]:
LINK_PATH = DATA_PATH / "links.csv"
MOVIES_PATH = DATA_PATH / "movies.csv"
RATINGS_PATH = DATA_PATH / "ratings.csv"
TAGS_PATH = DATA_PATH / "tags.csv"

links = pd.read_csv(LINK_PATH)
movies = pd.read_csv(MOVIES_PATH)
ratings = pd.read_csv(RATINGS_PATH)
tags = pd.read_csv(TAGS_PATH)

print("Links path:", LINK_PATH)
print("Movies path:", MOVIES_PATH)
print("Ratings path:", RATINGS_PATH)
print("Tags path:", TAGS_PATH)

Links path: data\ml-latest-small\links.csv
Movies path: data\ml-latest-small\movies.csv
Ratings path: data\ml-latest-small\ratings.csv
Tags path: data\ml-latest-small\tags.csv


## Inspect files

In [12]:
print("Links shape:", links.shape)
print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)
print("Tags shape:", tags.shape)

Links shape: (9742, 3)
Movies shape: (9742, 3)
Ratings shape: (100836, 4)
Tags shape: (3683, 4)


In [14]:
display(links.head())
display(movies.head())
display(ratings.head())
display(tags.head())

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


### Info
The most important file in this dataset is **`ratings.csv`**, as it contains information about users, movies, and their ratings. We will mainly use this file to build our recommendation system.

# 2. Evaluate Data Analysis

In [15]:
print("Number of users:", ratings["userId"].nunique())
print("Number of movies:", ratings["movieId"].nunique())
print("Number of ratings:", len(ratings))

print("\nRating distribution:")
print(ratings["rating"].value_counts().sort_index())

Number of users: 610
Number of movies: 9724
Number of ratings: 100836

Rating distribution:
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


### Observation
The dataset contains **610 users**, **9,724 movies**, and over **100,000 ratings**. Most ratings are between **3.0 and 4.0**, with **4.0 being the most common rating**. This suggests that users generally tend to give relatively positive ratings.

In [20]:
ratings_per_user = ratings.groupby("userId")["movieId"].count()

print("Average number of ratings per user:", int(ratings_per_user.mean()))
print("Minimum:", ratings_per_user.min())
print("Maximum:", ratings_per_user.max())

Average number of ratings per user: 165
Minimum: 20
Maximum: 2698


### Observation
On average, each user rated **165 movies**. However, there is a large difference in user activity — the least active users rated **20 movies**, while the most active user rated **2,698 movies**. This shows that some users are much more engaged with the platform than others.

In [23]:
movie_stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"])

popular_movies = movie_stats[movie_stats["count"] >= 50]

top_3_movies = popular_movies.sort_values(
    by="mean",
    ascending=False
).head(3)

top_3_movies = top_3_movies.merge(
    movies[["movieId", "title"]],
    on="movieId"
)

print(top_3_movies[["title", "mean", "count"]])

                              title      mean  count
0  Shawshank Redemption, The (1994)  4.429022    317
1             Godfather, The (1972)  4.289062    192
2                 Fight Club (1999)  4.272936    218


### Observation
The top 3 highest-rated movies are all **classic and highly acclaimed films**: *The Shawshank Redemption*, *The Godfather*, and *Fight Club*. This shows that these movies received consistently high ratings from a large number of users.

# 3. Data Preprocessing

## Create User x Movie matrix

In [29]:
user_movie_matrix = ratings.pivot(
    index="userId",
    columns="movieId",
    values="rating"
)

print(f"{user_movie_matrix.shape[0]} Users x {user_movie_matrix.shape[1]} Movies")
display(user_movie_matrix.head())

610 Users x 9724 Movies


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
total_cells = user_movie_matrix.size
missing_cells = user_movie_matrix.isna().sum().sum()
sparsity = missing_cells / total_cells * 100

print("Total cells:", total_cells)
print("Missing cells:", missing_cells)
print(f"Sparsity: {round(sparsity, 2)}%")

Total cells: 5931640
Missing cells: 5830804
Sparsity: 98.3%


### Observation
The User × Movie Matrix is highly sparse, with **98.3% of the cells containing missing values**. This means that users have rated only a small fraction of all available movies, which is typical for recommendation systems.

## Transpose the matrix from User × Movie to Movie × User and fill NaN with 0s.

In [46]:
movie_user_matrix = user_movie_matrix.T
movie_user_matrix = movie_user_matrix.fillna(0)

print(movie_user_matrix.shape)
print(movie_user_matrix.head(5))

(9724, 610)
userId   1    2    3    4    5    6    7    8    9    10   ...  601  602  603  \
movieId                                                    ...                  
1        4.0  0.0  0.0  0.0  4.0  0.0  4.5  0.0  0.0  0.0  ...  4.0  0.0  4.0   
2        0.0  0.0  0.0  0.0  0.0  4.0  0.0  4.0  0.0  0.0  ...  0.0  4.0  0.0   
3        4.0  0.0  0.0  0.0  0.0  5.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
4        0.0  0.0  0.0  0.0  0.0  3.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   
5        0.0  0.0  0.0  0.0  0.0  5.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0   

userId   604  605  606  607  608  609  610  
movieId                                     
1        3.0  4.0  2.5  4.0  2.5  3.0  5.0  
2        5.0  3.5  0.0  0.0  2.0  0.0  0.0  
3        0.0  0.0  0.0  0.0  2.0  0.0  0.0  
4        0.0  0.0  0.0  0.0  0.0  0.0  0.0  
5        3.0  0.0  0.0  0.0  0.0  0.0  0.0  

[5 rows x 610 columns]


# 4. Build Recommendation System with Collaborative Filtering

In [48]:
similarity_matrix = cosine_similarity(movie_user_matrix)

### Info
Hmm, I thought the model training would take longer. In this case, there is no traditional training process — we simply calculate the similarity between movies based on users' ratings.